In [5]:
import pandas as pd

# Step 1: Load the file
df = pd.read_excel("nadil_category_expenses.xlsx")

# Step 2: Fix the 'Date' column
df['Date'] = pd.to_datetime(df['Date'])

# Step 3: Clean and convert Payments and Receipts
df['Payments'] = pd.to_numeric(df['Payments'], errors='coerce').fillna(0.0)
df['Receipts'] = pd.to_numeric(df['Receipts'], errors='coerce').fillna(0.0)

# Step 4: Clean 'Balance' column
# Remove commas and non-numeric characters, handle weird suffixes like "B"
df['Balance'] = (
    df['Balance'].astype(str)                             # ensure string
    .str.replace(",", "", regex=False)                    # remove commas
    .str.replace(r"[^\d\.\-]", "", regex=True)            # remove non-numeric junk (e.g. "B")
    .replace("", pd.NA)                                   # empty strings to NA
)

# Now convert cleaned strings to float and forward-fill missing values
df['Balance'] = pd.to_numeric(df['Balance'], errors='coerce')
df['Balance'] = df['Balance'].fillna(method='ffill')

# Step 5: Convert 'Cluster' column to integer, fill NaNs with -1
df['Cluster'] = pd.to_numeric(df['Cluster'], errors='coerce').fillna(-1).astype(int)

# Done! Check your dtypes
print(df.dtypes)


Date                   datetime64[ns]
Discription                    object
Payments                      float64
Receipts                      float64
Balance                       float64
cleaned_particulars            object
Category                       object
Cluster                         int64
dtype: object


C:\Users\isuru\AppData\Local\Temp\ipykernel_24396\1448117084.py:24: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Balance'] = df['Balance'].fillna(method='ffill')


In [6]:
import pandas as pd

# Step 1: Load the Excel file
df = pd.read_excel("nadil_category_expenses.xlsx")

# Step 2: Ensure 'Date' is datetime and fill NaNs
df['Date'] = pd.to_datetime(df['Date'])
df['Payments'] = df['Payments'].fillna(0)
df['Receipts'] = df['Receipts'].fillna(0)
df['Balance'] = df['Balance'].astype(float)

# Step 3: Create a daily date range
date_range = pd.date_range(start=df['Date'].min(), end=df['Date'].max(), freq='D')

# Step 4: Get unique clusters
unique_clusters = df['Cluster'].unique()

# Step 5: Create all combinations of dates and clusters
full_index = pd.MultiIndex.from_product([date_range, unique_clusters], names=['Date', 'Cluster'])
full_df = pd.DataFrame(index=full_index).reset_index()

# Step 6: Aggregate original data by Date and Cluster
agg = df.groupby(['Date', 'Cluster'], as_index=False).agg({
    'Payments': 'sum',
    'Receipts': 'sum'
})

# Step 7: Merge with full date-cluster grid
merged_df = pd.merge(full_df, agg, on=['Date', 'Cluster'], how='left')
merged_df['Payments'] = merged_df['Payments'].fillna(0)
merged_df['Receipts'] = merged_df['Receipts'].fillna(0)

# Step 8: Calculate Net flow
merged_df['Net'] = merged_df['Receipts'] - merged_df['Payments']

# Step 9: Calculate daily net to update balance
daily_net = merged_df.groupby('Date')['Net'].sum().reset_index()

# Step 10: Get starting balance
starting_balance = df.sort_values('Date').iloc[0]['Balance']
daily_net['Balance'] = starting_balance + daily_net['Net'].cumsum()

# Step 11: Merge balance back to the full dataframe
final_df = pd.merge(merged_df.drop(columns='Net'), daily_net[['Date', 'Balance']], on='Date', how='left')

# Step 12: Final sanity check output
final_df.head(10)
--------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
Cell In[3], line 10
      8 df['Payments'] = df['Payments'].fillna(0)
      9 df['Receipts'] = df['Receipts'].fillna(0)
---> 10 df['Balance'] = df['Balance'].astype(float)
     12 # Step 3: Create a daily date range
     13 date_range = pd.date_range(start=df['Date'].min(), end=df['Date'].max(), freq='D')

File D:\my_notebook\code_practice_notebook\.venv\Lib\site-packages\pandas\core\generic.py:6643, in NDFrame.astype(self, dtype, copy, errors)
   6637     results = [
   6638         ser.astype(dtype, copy=copy, errors=errors) for _, ser in self.items()
   6639     ]
   6641 else:
   6642     # else, only a single dtype is given
-> 6643     new_data = self._mgr.astype(dtype=dtype, copy=copy, errors=errors)
   6644     res = self._constructor_from_mgr(new_data, axes=new_data.axes)
   6645     return res.__finalize__(self, method="astype")

File D:\my_notebook\code_practice_notebook\.venv\Lib\site-packages\pandas\core\internals\managers.py:430, in BaseBlockManager.astype(self, dtype, copy, errors)
    427 elif using_copy_on_write():
    428     copy = False
--> 430 return self.apply(
    431     "astype",
    432     dtype=dtype,
    433     copy=copy,
    434     errors=errors,
    435     using_cow=using_copy_on_write(),
    436 )

File D:\my_notebook\code_practice_notebook\.venv\Lib\site-packages\pandas\core\internals\managers.py:363, in BaseBlockManager.apply(self, f, align_keys, **kwargs)
    361         applied = b.apply(f, **kwargs)
    362     else:
--> 363         applied = getattr(b, f)(**kwargs)
    364     result_blocks = extend_blocks(applied, result_blocks)
    366 out = type(self).from_blocks(result_blocks, self.axes)

File D:\my_notebook\code_practice_notebook\.venv\Lib\site-packages\pandas\core\internals\blocks.py:758, in Block.astype(self, dtype, copy, errors, using_cow, squeeze)
    755         raise ValueError("Can not squeeze with more than one column.")
    756     values = values[0, :]  # type: ignore[call-overload]
--> 758 new_values = astype_array_safe(values, dtype, copy=copy, errors=errors)
    760 new_values = maybe_coerce_values(new_values)
    762 refs = None

File D:\my_notebook\code_practice_notebook\.venv\Lib\site-packages\pandas\core\dtypes\astype.py:237, in astype_array_safe(values, dtype, copy, errors)
    234     dtype = dtype.numpy_dtype
    236 try:
--> 237     new_values = astype_array(values, dtype, copy=copy)
    238 except (ValueError, TypeError):
    239     # e.g. _astype_nansafe can fail on object-dtype of strings
    240     #  trying to convert to float
    241     if errors == "ignore":

File D:\my_notebook\code_practice_notebook\.venv\Lib\site-packages\pandas\core\dtypes\astype.py:182, in astype_array(values, dtype, copy)
    179     values = values.astype(dtype, copy=copy)
    181 else:
--> 182     values = _astype_nansafe(values, dtype, copy=copy)
    184 # in pandas we don't store numpy str dtypes, so convert to object
    185 if isinstance(dtype, np.dtype) and issubclass(values.dtype.type, str):

File D:\my_notebook\code_practice_notebook\.venv\Lib\site-packages\pandas\core\dtypes\astype.py:133, in _astype_nansafe(arr, dtype, copy, skipna)
    129     raise ValueError(msg)
    131 if copy or arr.dtype == object or dtype == object:
    132     # Explicit copy, or required since NumPy can't view from / to object.
--> 133     return arr.astype(dtype, copy=True)
    135 return arr.astype(dtype, copy=copy)

ValueError: could not convert string to float: '13,430.5B'

SyntaxError: invalid syntax (2787770591.py, line 48)

In [7]:
import pandas as pd

# Step 1: Load the Excel file
df = pd.read_excel("nadil_category_expenses.xlsx")

# Step 2: Ensure 'Date' is datetime and fill NaNs
df['Date'] = pd.to_datetime(df['Date'])
df['Payments'] = df['Payments'].fillna(0)
df['Receipts'] = df['Receipts'].fillna(0)
df['Balance'] = df['Balance'].astype(float)

# Step 3: Create a daily date range
date_range = pd.date_range(start=df['Date'].min(), end=df['Date'].max(), freq='D')

# Step 4: Get unique clusters
unique_clusters = df['Cluster'].unique()

# Step 5: Create all combinations of dates and clusters
full_index = pd.MultiIndex.from_product([date_range, unique_clusters], names=['Date', 'Cluster'])
full_df = pd.DataFrame(index=full_index).reset_index()

# Step 6: Aggregate original data by Date and Cluster
agg = df.groupby(['Date', 'Cluster'], as_index=False).agg({
    'Payments': 'sum',
    'Receipts': 'sum'
})

# Step 7: Merge with full date-cluster grid
merged_df = pd.merge(full_df, agg, on=['Date', 'Cluster'], how='left')
merged_df['Payments'] = merged_df['Payments'].fillna(0)
merged_df['Receipts'] = merged_df['Receipts'].fillna(0)

# Step 8: Calculate Net flow
merged_df['Net'] = merged_df['Receipts'] - merged_df['Payments']

# Step 9: Calculate daily net to update balance
daily_net = merged_df.groupby('Date')['Net'].sum().reset_index()

# Step 10: Get starting balance
starting_balance = df.sort_values('Date').iloc[0]['Balance']
daily_net['Balance'] = starting_balance + daily_net['Net'].cumsum()

# Step 11: Merge balance back to the full dataframe
final_df = pd.merge(merged_df.drop(columns='Net'), daily_net[['Date', 'Balance']], on='Date', how='left')

# Step 12: Final sanity check output
final_df.head(10)


ValueError: could not convert string to float: '13,430.5B'

In [10]:
import pandas as pd

# Step 1: Load the Excel file
file_path = "nadil_category_expenses.xlsx"
df = pd.read_excel(file_path)

# Step 2: Ensure correct data types
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Payments'] = pd.to_numeric(df['Payments'], errors='coerce').fillna(0.0)
df['Receipts'] = pd.to_numeric(df['Receipts'], errors='coerce').fillna(0.0)

# Clean 'Balance' column: remove commas, non-numeric chars, then convert
df['Balance'] = (
    df['Balance'].astype(str)
    .str.replace(",", "", regex=False)
    .str.replace(r"[^\d\.\-]", "", regex=True)
    .replace("", pd.NA)
)
df['Balance'] = pd.to_numeric(df['Balance'], errors='coerce')
df['Balance'] = df['Balance'].fillna(method='ffill')

# Ensure 'Cluster' is int, fill NaNs with -1
df['Cluster'] = pd.to_numeric(df['Cluster'], errors='coerce').fillna(-1).astype(int)

# Step 3: Create date range from min to max date
date_range = pd.date_range(start=df['Date'].min(), end=df['Date'].max(), freq='D')

# Step 4: Get all unique clusters
all_clusters = sorted(df['Cluster'].unique())

# Step 5: Create full index of all date-cluster combinations
full_index = pd.MultiIndex.from_product([date_range, all_clusters], names=["Date", "Cluster"])
full_df = pd.DataFrame(index=full_index).reset_index()

# Step 6: Group original data to get daily sums per cluster
agg_df = df.groupby(['Date', 'Cluster'], as_index=False).agg({
    'Payments': 'sum',
    'Receipts': 'sum'
})

# Step 7: Merge with full date-cluster frame
merged = pd.merge(full_df, agg_df, on=['Date', 'Cluster'], how='left')
merged['Payments'] = merged['Payments'].fillna(0.0)
merged['Receipts'] = merged['Receipts'].fillna(0.0)

# Step 8: Calculate Net change
merged['Net'] = merged['Receipts'] - merged['Payments']

# Step 9: Compute daily net for entire day (not per cluster)
daily_net = merged.groupby('Date')['Net'].sum().cumsum()

# Step 10: Initialize balance
initial_balance = df.sort_values('Date').iloc[0]['Balance']
daily_balance = initial_balance + daily_net

# Step 11: Merge balance into merged dataframe
merged = merged.merge(daily_balance.rename("Balance"), on="Date", how="left")

# Step 12: Final output
final_df = merged[['Date', 'Cluster', 'Payments', 'Receipts', 'Balance']]
final_df.head(10)  # Show top 10 rows as preview



C:\Users\isuru\AppData\Local\Temp\ipykernel_24396\426910583.py:20: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Balance'] = df['Balance'].fillna(method='ffill')


,Date,Cluster,Payments,Receipts,Balance
0,2022-11-06,-1,0.0,0.0,3394.64
1,2022-11-06,0,6030.0,0.0,3394.64
2,2022-11-06,1,3030.0,0.0,3394.64
3,2022-11-06,2,0.0,0.0,3394.64
4,2022-11-06,3,0.0,0.0,3394.64
5,2022-11-06,4,0.0,0.0,3394.64
6,2022-11-06,5,0.0,0.0,3394.64
7,2022-11-06,6,0.0,0.0,3394.64
8,2022-11-06,7,0.0,0.0,3394.64
9,2022-11-06,8,0.0,0.0,3394.64


In [11]:
print(final_df.head())

        Date  Cluster  Payments  Receipts  Balance
0 2022-11-06       -1       0.0       0.0  3394.64
1 2022-11-06        0    6030.0       0.0  3394.64
2 2022-11-06        1    3030.0       0.0  3394.64
3 2022-11-06        2       0.0       0.0  3394.64
4 2022-11-06        3       0.0       0.0  3394.64


In [12]:
import pandas as pd

# Step 1: Read the Excel file
df = pd.read_excel('nadil_category_expenses.xlsx')

# Step 2: Clean data types
df['Date'] = pd.to_datetime(df['Date'])
df['Payments'] = pd.to_numeric(df['Payments'], errors='coerce').fillna(0)
df['Receipts'] = pd.to_numeric(df['Receipts'], errors='coerce').fillna(0)
df['Balance'] = df['Balance'].astype(str).str.replace(',', '').str.replace('B', '', regex=False)
df['Balance'] = pd.to_numeric(df['Balance'], errors='coerce')

# Step 3: Group by Date and Cluster
agg_df = df.groupby(['Date', 'Cluster'], as_index=False)[['Payments', 'Receipts']].sum()

# Step 4: Generate full date-cluster grid
all_dates = pd.date_range(df['Date'].min(), df['Date'].max())
all_clusters = df['Cluster'].unique()
full_index = pd.MultiIndex.from_product([all_dates, all_clusters], names=['Date', 'Cluster'])

# Step 5: Reindex the aggregated data to have all date-cluster combinations
agg_df = agg_df.set_index(['Date', 'Cluster']).reindex(full_index, fill_value=0).reset_index()

# Step 6: Compute daily total Net
daily_net = agg_df.groupby('Date')[['Payments', 'Receipts']].sum()
daily_net['Net'] = daily_net['Receipts'] - daily_net['Payments']
daily_net['Balance'] = daily_net['Net'].cumsum() + df.loc[df['Date'] == df['Date'].min(), 'Balance'].iloc[0]

# Step 7: Merge daily balance back to full DataFrame
agg_df = agg_df.merge(daily_net['Balance'], on='Date', how='left')

# Step 8: Final clean DataFrame
final_df = agg_df[['Date', 'Cluster', 'Payments', 'Receipts', 'Balance']]

# Optional: sort and reset index
final_df = final_df.sort_values(by=['Date', 'Cluster']).reset_index(drop=True)

# Show result
print(final_df.head(10))


        Date  Cluster  Payments  Receipts  Balance
0 2022-11-06       -1       0.0       0.0  3394.64
1 2022-11-06        0    6030.0       0.0  3394.64
2 2022-11-06        1    3030.0       0.0  3394.64
3 2022-11-06        2       0.0       0.0  3394.64
4 2022-11-06        3       0.0       0.0  3394.64
5 2022-11-06        4       0.0       0.0  3394.64
6 2022-11-06        5       0.0       0.0  3394.64
7 2022-11-06        6       0.0       0.0  3394.64
8 2022-11-06        7       0.0       0.0  3394.64
9 2022-11-06        8       0.0       0.0  3394.64


In [13]:
import pandas as pd

# Step 1: Load Excel
df = pd.read_excel('nadil_category_expenses.xlsx')

# Step 2: Clean data types
df['Date'] = pd.to_datetime(df['Date'])
df['Payments'] = pd.to_numeric(df['Payments'], errors='coerce').fillna(0)
df['Receipts'] = pd.to_numeric(df['Receipts'], errors='coerce').fillna(0)
df['Balance'] = df['Balance'].astype(str).str.replace(',', '').str.replace('B', '', regex=False)
df['Balance'] = pd.to_numeric(df['Balance'], errors='coerce')

# Step 3: Group payments/receipts by Date + Cluster
agg_df = df.groupby(['Date', 'Cluster'], as_index=False)[['Payments', 'Receipts']].sum()

# Step 4: Create full (date, cluster) index
all_dates = pd.date_range(df['Date'].min(), df['Date'].max())
all_clusters = df['Cluster'].unique()
full_index = pd.MultiIndex.from_product([all_dates, all_clusters], names=['Date', 'Cluster'])

# Step 5: Fill missing combinations with zeros
agg_df = agg_df.set_index(['Date', 'Cluster']).reindex(full_index, fill_value=0).reset_index()

# Step 6: Compute Net and daily balance
daily_totals = agg_df.groupby('Date')[['Payments', 'Receipts']].sum()
daily_totals['Net'] = daily_totals['Receipts'] - daily_totals['Payments']

# Starting balance from first available record
initial_balance = df.loc[df['Date'] == df['Date'].min(), 'Balance'].dropna().iloc[0]
daily_totals['Balance'] = daily_totals['Net'].cumsum() + initial_balance
daily_totals = daily_totals[['Balance']]  # We only need Balance now

# Step 7: Merge daily Balance back to full date-cluster table
agg_df = agg_df.merge(daily_totals, on='Date', how='left')

# Step 8: Final DataFrame
final_df = agg_df[['Date', 'Cluster', 'Payments', 'Receipts', 'Balance']]
final_df = final_df.sort_values(by=['Date', 'Cluster']).reset_index(drop=True)

# Show output
print(final_df.head(15))


         Date  Cluster  Payments  Receipts  Balance
0  2022-11-06       -1       0.0       0.0  3394.64
1  2022-11-06        0    6030.0       0.0  3394.64
2  2022-11-06        1    3030.0       0.0  3394.64
3  2022-11-06        2       0.0       0.0  3394.64
4  2022-11-06        3       0.0       0.0  3394.64
5  2022-11-06        4       0.0       0.0  3394.64
6  2022-11-06        5       0.0       0.0  3394.64
7  2022-11-06        6       0.0       0.0  3394.64
8  2022-11-06        7       0.0       0.0  3394.64
9  2022-11-06        8       0.0       0.0  3394.64
10 2022-11-06        9       0.0       0.0  3394.64
11 2022-11-06       10       0.0       0.0  3394.64
12 2022-11-06       11       0.0       0.0  3394.64
13 2022-11-07       -1       0.0       0.0  3394.64
14 2022-11-07        0       0.0       0.0  3394.64


In [14]:
import pandas as pd

# Step 1: Load Excel
df = pd.read_excel('nadil_category_expenses.xlsx')

# Step 2: Clean data types
df['Date'] = pd.to_datetime(df['Date'])
df['Payments'] = pd.to_numeric(df['Payments'], errors='coerce').fillna(0)
df['Receipts'] = pd.to_numeric(df['Receipts'], errors='coerce').fillna(0)
df['Balance'] = df['Balance'].astype(str).str.replace(',', '').str.replace('B', '', regex=False)
df['Balance'] = pd.to_numeric(df['Balance'], errors='coerce')

# Step 3: Group payments/receipts by Date + Cluster
agg_df = df.groupby(['Date', 'Cluster'], as_index=False)[['Payments', 'Receipts']].sum()

# Step 4: Create full (date, cluster) index
all_dates = pd.date_range(df['Date'].min(), df['Date'].max())
all_clusters = sorted(df['Cluster'].unique())
full_index = pd.MultiIndex.from_product([all_dates, all_clusters], names=['Date', 'Cluster'])

# Step 5: Fill missing combinations with zeros
agg_df = agg_df.set_index(['Date', 'Cluster']).reindex(full_index, fill_value=0).reset_index()

# Step 6: Compute Net per date and running Balance
daily_net = agg_df.groupby('Date')[['Payments', 'Receipts']].sum()
daily_net['Net'] = daily_net['Receipts'] - daily_net['Payments']

# Step 7: Starting balance (from first row of original df)
initial_balance = df.sort_values('Date')['Balance'].dropna().iloc[0]

# Step 8: Cumulative balance calculation
daily_net['Balance'] = initial_balance + daily_net['Net'].cumsum()
daily_net = daily_net[['Balance']]  # We just need Balance

# Step 9: Merge per-day balance back into agg_df
agg_df = agg_df.merge(daily_net, on='Date', how='left')

# Step 10: Final sorted DataFrame
final_df = agg_df[['Date', 'Cluster', 'Payments', 'Receipts', 'Balance']]
final_df = final_df.sort_values(['Date', 'Cluster']).reset_index(drop=True)

# Show result
print(final_df.head(20))


         Date  Cluster  Payments  Receipts  Balance
0  2022-11-06       -1       0.0       0.0  3394.64
1  2022-11-06        0    6030.0       0.0  3394.64
2  2022-11-06        1    3030.0       0.0  3394.64
3  2022-11-06        2       0.0       0.0  3394.64
4  2022-11-06        3       0.0       0.0  3394.64
5  2022-11-06        4       0.0       0.0  3394.64
6  2022-11-06        5       0.0       0.0  3394.64
7  2022-11-06        6       0.0       0.0  3394.64
8  2022-11-06        7       0.0       0.0  3394.64
9  2022-11-06        8       0.0       0.0  3394.64
10 2022-11-06        9       0.0       0.0  3394.64
11 2022-11-06       10       0.0       0.0  3394.64
12 2022-11-06       11       0.0       0.0  3394.64
13 2022-11-07       -1       0.0       0.0  3394.64
14 2022-11-07        0       0.0       0.0  3394.64
15 2022-11-07        1       0.0       0.0  3394.64
16 2022-11-07        2       0.0       0.0  3394.64
17 2022-11-07        3       0.0       0.0  3394.64
18 2022-11-0

In [15]:
import pandas as pd

# Step 1: Load Excel
df = pd.read_excel('nadil_category_expenses.xlsx')

# Step 2: Clean data types
df['Date'] = pd.to_datetime(df['Date'])
df['Payments'] = pd.to_numeric(df['Payments'], errors='coerce').fillna(0)
df['Receipts'] = pd.to_numeric(df['Receipts'], errors='coerce').fillna(0)

# Step 3: Remove existing 'Balance' column
df = df.drop(columns=['Balance'])

# Step 4: Initialize the balance with the first entry
initial_balance = df.sort_values('Date')['Balance'].dropna().iloc[0]

# Step 5: Create new 'Balance' column starting from the initial balance
df['Balance'] = initial_balance + (df['Receipts'] - df['Payments']).cumsum()

# Show result
print(df.head(20))


KeyError: 'Balance'

In [16]:
import pandas as pd

# Step 1: Load Excel file
df = pd.read_excel('nadil_category_expenses.xlsx')

# Step 2: Clean data types (ensure Payments and Receipts are numeric and Date is datetime)
df['Date'] = pd.to_datetime(df['Date'])
df['Payments'] = pd.to_numeric(df['Payments'], errors='coerce').fillna(0)
df['Receipts'] = pd.to_numeric(df['Receipts'], errors='coerce').fillna(0)

# Step 3: Initialize the balance with the first entry, assuming the first row is the starting balance
initial_balance = 3394.64  # If you have a different value, adjust it here

# Step 4: Calculate the new 'Balance' column using Payments and Receipts
df['Balance'] = initial_balance + (df['Receipts'] - df['Payments']).cumsum()

# Show the updated dataframe (first 20 rows for preview)
print(df.head(20))


         Date                Discription  Payments  Receipts   Balance  \
0  2022-11-06           t ahirt OTHBNK T   6030.00       0.0  -2635.36   
1  2022-11-06   010001088282101 OTHBNK T   3030.00       0.0  -5665.36   
2  2022-11-15  RIB/RMB SE.CH 20 IBMB Chg     25.00       0.0  -5690.36   
3  2022-11-18  nadil Siriwardha MB SA TF    450.00       0.0  -6140.36   
4  2022-12-24             nadil OTHBNK T   7530.00       0.0 -13670.36   
5  2022-12-26             nadil OTHBNK T   4030.00       0.0 -17700.36   
6  2022-12-28                    CSH WDR  10000.00       0.0 -27700.36   
7  2022-12-29                    CSH WDR   5000.00       0.0 -32700.36   
8  2023-01-06              film OTHBNK T   3130.00       0.0 -35830.36   
9  2023-01-08              film OTHBNK T   3030.00       0.0 -38860.36   
10 2023-01-22            tshirt OTHBNK T   5030.00       0.0 -43890.36   
11 2023-01-31                    CSH WDR  21200.00       0.0 -65090.36   
12 2023-01-31                     WH T

In [17]:
print(df.head())

        Date                Discription  Payments  Receipts   Balance  \
0 2022-11-06           t ahirt OTHBNK T    6030.0       0.0  -2635.36   
1 2022-11-06   010001088282101 OTHBNK T    3030.0       0.0  -5665.36   
2 2022-11-15  RIB/RMB SE.CH 20 IBMB Chg      25.0       0.0  -5690.36   
3 2022-11-18  nadil Siriwardha MB SA TF     450.0       0.0  -6140.36   
4 2022-12-24             nadil OTHBNK T    7530.0       0.0 -13670.36   

            cleaned_particulars                  Category  Cluster  
0              t ahirt othbnk t            NADIL OTHBNK T        0  
1      010001088282101 othbnk t                  OTHBNK T        1  
2  rib/rmb se.ch 20 ibmb charge  RIBRMB SECH  IBMB CHARGE        2  
3     nadil siriwardha mb sa tf            NADIL OTHBNK T        0  
4                nadil othbnk t            NADIL OTHBNK T        0  
